# Demo 3: Build one aggregating pivot

**Learning question:** How can we specify an aggregating pivot completely and prove that every populated cell matches the equivalent grouped result?

The input has grain **one recorded encounter per row**. The wide display has **one observed facility per row**, while each populated cell summarizes **one observed facility--service group**. This required demo is Colab-first and runs equivalently in local Jupyter or VS Code. Colab storage is ephemeral, and changes opened from GitHub are not automatically saved back to the repository.

Use only the supplied synthetic, non-identifying fixture. Do not add credentials, private records, manual uploads, or Drive mounts. Restart the kernel and run every cell in order; stored output is not execution evidence. Assignment Colab support remains conditional on the repository-save and Classroom50 pilot.


In [ ]:
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

PYTHON_CANDIDATE = "3.12.13"
NUMPY_CANDIDATE = "2.0.2"
PANDAS_CANDIDATE = "3.0.3"
COURSE_PACKAGES = {"numpy": NUMPY_CANDIDATE, "pandas": PANDAS_CANDIDATE}


def installed_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


mismatched = [
    f"{package_name}=={candidate}"
    for package_name, candidate in COURSE_PACKAGES.items()
    if installed_version(package_name) != candidate
]
if mismatched:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *mismatched]
    )

import numpy as np
import pandas as pd

assert platform.python_version() == PYTHON_CANDIDATE
assert np.__version__ == NUMPY_CANDIDATE
assert pd.__version__ == PANDAS_CANDIDATE
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)


## Define the pivot before calling it

A structural `pivot` from Lecture 06 rearranges unique row/column combinations and does not aggregate. An aggregating **`pivot_table`** groups repeated combinations and places their summaries across row and column axes.

Name all five required choices before execution:

- `index="facility"`: observed facilities become displayed rows;
- `columns="service"`: observed services become displayed columns;
- `values="charge"`: charge supplies the measurements;
- `aggfunc="mean"`: repeated facility--service rows become one mean; and
- `observed=True`: unused category levels such as Remote are omitted.

Predict North, South, and West as rows; Consult, Follow-up, and Procedure as columns; eight populated cells; and one missing South--Follow-up cell. Missing means no input row for that combination, not a measured zero.


In [ ]:
from hashlib import sha256
from pathlib import Path

EXPECTED_FIXTURE_SHA256 = "24a31904c1371553ff3af627dc21146ed743c8c0c47452ade3628c2fc199c5dc"
FIXTURE_BYTES = (
    b"encounter_id,facility,provider_id,service,charge,wait_minutes,rating\n"
    b"E001,North,P01,Consult,120,20,4\n"
    b"E002,North,P01,Follow-up,80,12,\n"
    b"E003,North,P02,Consult,150,30,5\n"
    b"E004,North,P02,Procedure,210,50,5\n"
    b"E005,South,P03,Consult,110,18,4\n"
    b"E006,South,P03,Consult,90,16,\n"
    b"E007,South,P04,Procedure,220,45,\n"
    b"E008,South,P04,Procedure,125,25,4\n"
    b"E009,West,P05,Consult,130,25,3\n"
    b"E010,West,P05,Procedure,200,40,4\n"
    b"E011,West,P06,Consult,140,35,3\n"
    b"E012,West,P06,Follow-up,75,15,4\n"
)
FACILITY_LEVELS = ["North", "South", "West", "Remote"]
SERVICE_LEVELS = ["Consult", "Follow-up", "Procedure"]


def find_demo_directory(start):
    current = start.resolve()
    while True:
        for candidate in (current, current / "08" / "demo"):
            if (
                (candidate / "DEMO_GUIDE.md").is_file()
                and (candidate / ".python-version").is_file()
            ):
                return candidate
        if current.parent == current:
            return None
        current = current.parent


DEMO_DIRECTORY = find_demo_directory(Path.cwd())
if DEMO_DIRECTORY is None:
    DEMO_DIRECTORY = Path.cwd().resolve()

DATA_DIRECTORY = DEMO_DIRECTORY / "data"
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
FIXTURE_PATH = DATA_DIRECTORY / "encounters.csv"
if not FIXTURE_PATH.exists():
    FIXTURE_PATH.write_bytes(FIXTURE_BYTES)

actual_fixture_sha256 = sha256(FIXTURE_PATH.read_bytes()).hexdigest()
assert actual_fixture_sha256 == EXPECTED_FIXTURE_SHA256, (
    "encounters.csv does not match the supplied fixture checksum. "
    "Restore the committed file; corrupt data are never replaced silently."
)

OUTPUT_DIRECTORY = DEMO_DIRECTORY / "output"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIRECTORY / "mean_charge_pivot.csv"
if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()


def write_repeatable_csv(frame, path, *, na_rep=""):
    frame.to_csv(
        path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        na_rep=na_rep,
    )
    first_bytes = path.read_bytes()
    frame.to_csv(
        path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        na_rep=na_rep,
    )
    assert path.read_bytes() == first_bytes
    return first_bytes


encounters = pd.read_csv(
    FIXTURE_PATH,
    dtype={
        "encounter_id": "string",
        "provider_id": "string",
        "charge": "int64",
        "wait_minutes": "int64",
        "rating": "Int64",
    },
)
encounters["facility"] = pd.Categorical(
    encounters["facility"],
    categories=FACILITY_LEVELS,
    ordered=True,
)
encounters["service"] = pd.Categorical(
    encounters["service"],
    categories=SERVICE_LEVELS,
    ordered=True,
)

assert encounters.shape == (12, 7)
assert encounters["encounter_id"].is_unique
assert encounters["encounter_id"].dtype == pd.StringDtype()
assert encounters["provider_id"].dtype == pd.StringDtype()
assert encounters["facility"].dtype == pd.CategoricalDtype(
    FACILITY_LEVELS, ordered=True
)
assert encounters["service"].dtype == pd.CategoricalDtype(
    SERVICE_LEVELS, ordered=True
)
assert encounters["charge"].dtype == np.dtype("int64")
assert encounters["wait_minutes"].dtype == np.dtype("int64")
assert encounters["rating"].dtype == pd.Int64Dtype()

print("Demo directory:", DEMO_DIRECTORY)
print("Fixture SHA-256:", actual_fixture_sha256)
print(encounters)


## Build an independent two-key reference

The reference GroupBy has output grain one observed facility--service combination. It is built independently in this notebook so the pivot can be checked rather than trusted.


In [ ]:
grouped_mean_charge = (
    encounters.groupby(
        ["facility", "service"],
        as_index=False,
        observed=True,
        sort=True,
        dropna=True,
    )
    .agg(mean_charge=("charge", "mean"))
)

assert grouped_mean_charge.columns.tolist() == [
    "facility",
    "service",
    "mean_charge",
]
assert grouped_mean_charge.shape == (8, 3)
assert list(
    zip(
        grouped_mean_charge["facility"].astype("string"),
        grouped_mean_charge["service"].astype("string"),
        grouped_mean_charge["mean_charge"],
    )
) == [
    ("North", "Consult", 135.0),
    ("North", "Follow-up", 80.0),
    ("North", "Procedure", 210.0),
    ("South", "Consult", 100.0),
    ("South", "Procedure", 172.5),
    ("West", "Consult", 135.0),
    ("West", "Follow-up", 75.0),
    ("West", "Procedure", 200.0),
]

print(grouped_mean_charge)


## Build and verify exactly one pivot table

The row grain of the displayed table is one observed facility. Each populated cell is more specific: it summarizes one observed facility--service group. Every populated cell must equal its GroupBy reference, and the only absent combination must remain missing rather than become zero.


In [ ]:
mean_charge_pivot = pd.pivot_table(
    encounters,
    index="facility",
    columns="service",
    values="charge",
    aggfunc="mean",
    observed=True,
    sort=True,
    dropna=True,
)

assert mean_charge_pivot.index.name == "facility"
assert mean_charge_pivot.columns.name == "service"
assert mean_charge_pivot.index.astype("string").tolist() == [
    "North",
    "South",
    "West",
]
assert mean_charge_pivot.columns.astype("string").tolist() == [
    "Consult",
    "Follow-up",
    "Procedure",
]
assert mean_charge_pivot.shape == (3, 3)
assert mean_charge_pivot.loc["North", "Consult"] == 135.0
assert mean_charge_pivot.loc["South", "Procedure"] == 172.5
assert pd.isna(mean_charge_pivot.loc["South", "Follow-up"])
assert "Remote" not in mean_charge_pivot.index.astype("string")

populated_cell_count = int(mean_charge_pivot.notna().to_numpy().sum())
missing_cell_count = int(mean_charge_pivot.isna().to_numpy().sum())
assert populated_cell_count == 8
assert missing_cell_count == 1

for grouped_row in grouped_mean_charge.itertuples(index=False):
    pivot_value = mean_charge_pivot.loc[
        grouped_row.facility,
        grouped_row.service,
    ]
    assert np.isclose(pivot_value, grouped_row.mean_charge)

assert populated_cell_count == len(grouped_mean_charge)

pivot_for_csv = (
    mean_charge_pivot.rename_axis(columns=None)
    .reset_index()
)
assert pivot_for_csv.columns.tolist() == [
    "facility",
    "Consult",
    "Follow-up",
    "Procedure",
]

pivot_bytes = write_repeatable_csv(
    pivot_for_csv,
    OUTPUT_PATH,
    na_rep="",
)
EXPECTED_PIVOT_BYTES = (
    b"facility,Consult,Follow-up,Procedure\n"
    b"North,135.0,80.0,210.0\n"
    b"South,100.0,,172.5\n"
    b"West,135.0,75.0,200.0\n"
)
assert pivot_bytes == EXPECTED_PIVOT_BYTES

pivot_readback = pd.read_csv(
    OUTPUT_PATH,
    dtype={
        "facility": "string",
        "Consult": "float64",
        "Follow-up": "float64",
        "Procedure": "float64",
    },
)
expected_pivot_readback = pivot_for_csv.copy()
expected_pivot_readback["facility"] = expected_pivot_readback[
    "facility"
].astype("string")
pd.testing.assert_frame_equal(
    pivot_readback,
    expected_pivot_readback,
)
assert pivot_readback["Follow-up"].isna().sum() == 1
assert pd.isna(
    pivot_readback.loc[
        pivot_readback["facility"].eq("South"),
        "Follow-up",
    ].iloc[0]
)
assert not (pivot_readback[["Consult", "Follow-up", "Procedure"]] == 0).any().any()

demo3_verified = True
print(mean_charge_pivot)
print("Wrote:", OUTPUT_PATH)


In [ ]:
assert demo3_verified is True
assert sha256(FIXTURE_PATH.read_bytes()).hexdigest() == EXPECTED_FIXTURE_SHA256
assert grouped_mean_charge.shape == (8, 3)
assert mean_charge_pivot.shape == (3, 3)
assert populated_cell_count == 8
assert missing_cell_count == 1
assert OUTPUT_PATH.is_file()
assert OUTPUT_PATH.read_bytes() == EXPECTED_PIVOT_BYTES
print("Lecture 08 Demo 3 fresh-execution verification passed.")
